# Trick or Data: Statistical Analysis

## Objective

This notebook investigates the statistical significance and magnitude of relationships identified during the exploratory data analysis.

The analysis focuses on comparing candy popularity across different attributes and evaluating whether the observed differences are supported by statistical evidence.

## Research Questions

1. Is the difference in popularity between chocolate and non-chocolate candies statistically significant?
2. Is the difference between fruity and non-fruity candies statistically significant?
3. How large are the observed differences between these groups?
4. Which candy attributes show the strongest associations with popularity?

## Statistical Methods

The analysis explores:

- Null and alternative hypotheses
- Independent samples t-tests
- Mann–Whitney U tests
- p-values and statistical significance
- Confidence intervals
- Effect sizes

## Dataset

- **Source:** FiveThirtyEight Halloween Candy Power Ranking
- **Input:** Cleaned candy dataset
- **Observations:** 85 candy products
- **Target variable:** `winpercent`
- **Tools:** Python, Pandas, NumPy, SciPy, and Matplotlib

## 1. Data Loading

This section loads the cleaned dataset generated during the data preparation process.

The processed dataset serves as the input for the statistical analyses performed in this notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/Leticia-Sousa-DS/trick-or-data/refs/heads/main/data/processed/candy_data_cleaned.csv"

df = pd.read_csv(url)

In [2]:
print(f"Dataset shape: {df.shape}")
display(df.head())

Dataset shape: (85, 13)


,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent
0,100 Grand,1,0,1,0,0,1,0,1,0,0.732,0.860,66.971725
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.604,0.511,67.602936
2,One dime,0,0,0,0,0,0,0,0,0,0.011,0.116,32.261086
3,One quarter,0,0,0,0,0,0,0,0,0,0.011,0.511,46.116505
4,Air Heads,0,1,0,0,0,0,0,0,0,0.906,0.511,52.341465


## 2. Hypothesis Formulation

This section defines the null and alternative hypotheses for the statistical comparisons.

### Research Question 1

Is there a statistically significant difference in popularity between chocolate and non-chocolate candies?

- **Null hypothesis (H0):** There is no difference in mean win percentage between chocolate and non-chocolate candies.
- **Alternative hypothesis (H1):** There is a difference in mean win percentage between chocolate and non-chocolate candies.

The test will use a two-sided alternative hypothesis, since the objective is to determine whether a difference exists, regardless of its direction.

## 3. Descriptive Statistics

This section examines the descriptive statistics of chocolate and non-chocolate candies to understand differences in their popularity distributions before conducting hypothesis tests.

The analysis includes sample size, mean, standard deviation, median, minimum, and maximum win percentages.

In [3]:
df.groupby('chocolate')['winpercent'].agg(
    ['count', 'mean', 'std', 'median', 'min', 'max']
)

,count,mean,std,median,min,max
chocolate,,,,,,
0,48,42.142257,10.220959,41.646933,22.445341,67.037628
1,37,60.921529,12.811124,60.800701,34.722000,84.180290


## 4. Statistical Test Selection

This section defines the statistical methods used to evaluate the hypotheses formulated in the previous section.

The Welch's t-test is selected as the primary method to compare mean win percentages between chocolate and non-chocolate candies.

This test compares two independent groups without assuming equal population variances.

The Mann–Whitney U test is also used as a complementary non-parametric method to compare the distributions of the two groups.

Both tests use a two-sided alternative hypothesis, as the objective is to investigate whether a difference exists, regardless of its direction.

In [5]:
# Separate candies into chocolate and non-chocolate groups
chocolate_group = df[df['chocolate'] == 1]['winpercent']
non_chocolate_group = df[df['chocolate'] == 0]['winpercent']

# Welch's t-test
welch_result = stats.ttest_ind(
    chocolate_group,
    non_chocolate_group,
    equal_var=False,
    alternative='two-sided'
)

# Mann–Whitney U test
mannwhitney_result = stats.mannwhitneyu(
    chocolate_group,
    non_chocolate_group,
    alternative='two-sided'
)



Welch's t-test
t-statistic: 7.3031
p-value: 0.000000

Mann–Whitney U test
U-statistic: 1542.0000
p-value: 0.000000

P-values in scientific notation
Welch's t-test: 4.1635072177e-10
Mann–Whitney U test: 6.9349267098e-09


## 5. Statistical Test Results

Both statistical tests indicate a statistically significant difference between chocolate and non-chocolate candies.

The Welch's t-test evaluates the difference in mean popularity, while the Mann–Whitney U test provides a complementary comparison based on the ranks of the observations.

The results provide evidence against the null hypothesis at the 5% significance level.

However, statistical significance does not necessarily indicate the practical magnitude of a difference, nor does it establish a causal relationship between chocolate content and candy popularity.

In [6]:
print("Welch's t-test")
print(f"t-statistic: {welch_result.statistic:.4f}")
print(f"p-value: {welch_result.pvalue:.6f}")

print("\nMann–Whitney U test")
print(f"U-statistic: {mannwhitney_result.statistic:.4f}")
print(f"p-value: {mannwhitney_result.pvalue:.6f}")

print("\nP-values in scientific notation")
print(f"Welch's t-test: {welch_result.pvalue:.10e}")
print(f"Mann–Whitney U test: {mannwhitney_result.pvalue:.10e}")

Welch's t-test
t-statistic: 7.3031
p-value: 0.000000

Mann–Whitney U test
U-statistic: 1542.0000
p-value: 0.000000

P-values in scientific notation
Welch's t-test: 4.1635072177e-10
Mann–Whitney U test: 6.9349267098e-09


## 6. Confidence Interval

This section estimates the difference in mean popularity between chocolate and non-chocolate candies using a 95% confidence interval.

The interval is calculated using the Welch-Satterthwaite method, which accounts for potentially unequal variances and sample sizes between the two groups.

The confidence interval provides a range of plausible values for the difference in population means under the assumptions of the statistical model.

In [7]:
# Calculate the difference between group means
mean_difference = chocolate_group.mean() - non_chocolate_group.mean()

# Calculate the standard error
standard_error = np.sqrt(
    (chocolate_group.var(ddof=1) / len(chocolate_group)) +
    (non_chocolate_group.var(ddof=1) / len(non_chocolate_group))
)

# Calculate Welch-Satterthwaite degrees of freedom
numerator = (
    (chocolate_group.var(ddof=1) / len(chocolate_group)) +
    (non_chocolate_group.var(ddof=1) / len(non_chocolate_group))
) ** 2

denominator = (
    (chocolate_group.var(ddof=1) / len(chocolate_group)) ** 2 /
    (len(chocolate_group) - 1)
) + (
    (non_chocolate_group.var(ddof=1) / len(non_chocolate_group)) ** 2 /
    (len(non_chocolate_group) - 1)
)

degrees_of_freedom = numerator / denominator

# Calculate the 95% confidence interval
confidence_level = 0.95
alpha = 1 - confidence_level

critical_value = stats.t.ppf(
    1 - alpha / 2,
    degrees_of_freedom
)

margin_of_error = critical_value * standard_error

lower_bound = mean_difference - margin_of_error
upper_bound = mean_difference + margin_of_error

print(f"Mean difference: {mean_difference:.4f}")
print(f"Standard error: {standard_error:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom:.2f}")
print(f"95% confidence interval: [{lower_bound:.4f}, {upper_bound:.4f}]")

Mean difference: 18.7793
Standard error: 2.5714
Degrees of freedom: 67.54
95% confidence interval: [13.6474, 23.9111]


### Confidence Interval Interpretation

The estimated mean difference in popularity between chocolate and non-chocolate candies is approximately 18.78 percentage points.

The 95% confidence interval ranges from 13.65 to 23.91 percentage points. Since the interval does not include zero, it is consistent with the statistically significant result obtained from Welch's t-test.

This suggests that chocolate candies have a higher mean win percentage in the analyzed dataset.

The confidence interval quantifies uncertainty around the estimated difference, but it does not establish a causal relationship between chocolate content and popularity.